In [0]:
from pyspark.sql import functions as F

catalogo = "cambio"

df_silver = spark.table(f"{catalogo}.silver.cotacao_moeda")

df_gold = (
    df_silver
    .groupBy(
        "codigo_moeda",
        "data_cotacao"
    )
    .agg(
        F.count("*").alias("qtd_cotacoes"),

        F.avg("cotacao_compra").cast("decimal(18,6)").alias("media_compra"),
        F.avg("cotacao_venda").cast("decimal(18,6)").alias("media_venda"),

        F.min("cotacao_compra").alias("menor_compra"),
        F.max("cotacao_compra").alias("maior_compra"),

        F.min("cotacao_venda").alias("menor_venda"),
        F.max("cotacao_venda").alias("maior_venda")
    )
)

df_gold.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema","true") \
    .saveAsTable(f"{catalogo}.gold.fato_cotacao_diaria")